<a href="https://colab.research.google.com/github/kanish-27/KanishKrishna-Codeboosters-Internship-2026/blob/main/Phase_01_Data_Engineering/Day_03_ETL_Pandas_APIs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install requests --quiet
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')
print(f'pandas :{pd.__version__}')
print(f'requests :{requests.__version__}')

All libraries imported successfully!
pandas :2.2.2
requests :2.32.4


ETL is a data integration pattern. EXTRACT: pull data from heterogeneous sources (REST APIs, databases, flat files, streams). TRANSFORM: apply
business logic - handle nulls, remove duplicates, validate types, normalize formats, aggregate, enrich. LOAD: write the transformed data to a
target system (data warehouse, database, analytics platform). ETL ensures that downstream consumers (analysts, ML models, dashboards) always
receive clean, trusted data.

In [ ]:
df=pd.read_csv('/content/messy_sales_data.csv')
print(f'Columns:{df.shape[1]}')
print(f'Rows:{df.shape[0]}')

Columns:9
Rows:30


In [ ]:
print(df.columns.tolist())
df.head()

['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'order_date', 'city', 'sales_rep']


,order_id,customer_name,product,category,quantity,unit_price,order_date,city,sales_rep
0,1001,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma
1,1002,Priya Nair,NaN,Electronics,1.0,15000,2024-01-07,Delhi,Sunita Rao
2,1003,AMIT VERMA,Keyboard,Accessories,3.0,1200,2024-01-08,Bangalore,Anil Sharma
3,1004,Sunita Patel,Monitor,Electronics,NaN,22000,2024-01-10,Chennai,Ravi Kumar
4,1005,Ramesh Kumar,Laptop,Electronics,2.0,45000,2024-01-05,Mumbai,Anil Sharma


In [ ]:
print('='*55)
print(' DATA QUALITY DIAGONISIS REPORT ')
print('='*55)

print('\n[1] MISSING VALUES per column:')
print(df.isnull().sum())
print(f'\n[2] Duplicate ROWS :{df.duplicated().sum()}')
print('\n[3] DATA TYPES')
print(df.dtypes)
print('\n[4] UNIQUE CATEGORIES:',df['category'].unique())
print('\n[5] Sample customer names:',df['customer_name'].dropna().unique()[:8])
print('\n[6] Sample order_date values:',df['order_date'].unique()[:6])

 DATA QUALITY DIAGONISIS REPORT 

[1] MISSING VALUES per column:
order_id         0
customer_name    2
product          1
category         1
quantity         3
unit_price       0
order_date       0
city             0
sales_rep        0
dtype: int64

[2] Duplicate ROWS :0

[3] DATA TYPES
order_id           int64
customer_name     object
product           object
category          object
quantity         float64
unit_price         int64
order_date        object
city              object
sales_rep         object
dtype: object

[4] UNIQUE CATEGORIES: ['Electronics' 'Accessories' nan]

[5] Sample customer names: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh' 'Ananya Das' 'Vikram Iyer']

[6] Sample order_date values: ['2024-01-05' '2024-01-07' '2024-01-08' '2024-01-10' '07-01-2024'
 '2024-01-12']


In [ ]:
cdf=df.copy()
print(f'Working copy created :{df.shape}')
print('df is untouched - we can always reset by running df = df.copy()')

Working copy created :(30, 9)
df is untouched - we can always reset by running df = df.copy()


In [ ]:
print('Before fixing nulls:',cdf.isnull().sum().sum(),'total missing values')
cdf['customer_name'].fillna('Unkonwn Customer',inplace=True)
median_qty=cdf['quantity'].median()
cdf['quantity'].fillna(median_qty,inplace=True)
print(f' Filled missing quantity with median: {median_qty}')
cdf['category'].fillna('Uncategorized',inplace=True)
print('After fixing nulls:',df.isnull().sum().sum(),'total missing values')

Before fixing nulls: 1 total missing values
 Filled missing quantity with median: 2.0
After fixing nulls: 1 total missing values


In [ ]:
print(f'Before deduplication:{len(cdf)} rows')
print(f'Duplicate rows:{cdf.duplicated().sum()}')
print('\nDuplicate rows: ')
print(cdf[cdf.duplicated(keep=False)][['order_id','customer_name','product','order_date']])
cdf.drop_duplicates(inplace=True)
print(f'\nAfter duplication: {len(cdf)} rows')
print(f'Rows removed: {len(cdf)-len(df)}')

Before deduplication:30 rows
Duplicate rows:0

Duplicate rows: 
Empty DataFrame
Columns: [order_id, customer_name, product, order_date]
Index: []

After duplication: 30 rows
Rows removed: 0


In [ ]:
print('Sample dates before parsing:')
print(cdf['order_date'].head(8).tolist())
cdf['order_date']=pd.to_datetime(cdf['order_date'],dayfirst=False,errors='coerce')
nat_count=cdf['order_date'].isnull().sum()
print(f'\nUnparseable dates: (NAT): {nat_count}')
cdf['year'] = cdf['order_date'].dt.year
cdf['month']=cdf['order_date'].dt.month
cdf['month_name']=cdf['order_date'].dt.strftime('%B')
print('\nSample dates after parsing:')
print(cdf[['order_date','year','month','month_name']].head(6))

Sample dates before parsing:
[Timestamp('2024-01-05 00:00:00'), Timestamp('2024-01-07 00:00:00'), Timestamp('2024-01-08 00:00:00'), Timestamp('2024-01-10 00:00:00'), Timestamp('2024-01-05 00:00:00'), NaT, Timestamp('2024-01-12 00:00:00'), Timestamp('2024-01-13 00:00:00')]

Unparseable dates: (NAT): 2

Sample dates after parsing:
  order_date    year  month month_name
0 2024-01-05  2024.0    1.0    January
1 2024-01-07  2024.0    1.0    January
2 2024-01-08  2024.0    1.0    January
3 2024-01-10  2024.0    1.0    January
4 2024-01-05  2024.0    1.0    January
5        NaT     NaN    NaN        NaN


In [ ]:
print('Before standardization:',df['customer_name'].unique()[:6])
cdf['customer_name']=(
    cdf['customer_name']
    .str.strip()
    .str.title()
)
print(f"\n After Standardization:", cdf['customer_name'].unique()[:6])

print(f"\n Before Keyboard rows with electronics category:")
wrong_mask = (cdf['product']=='keyboard') & (cdf['category']=='Electronics')
print(cdf[wrong_mask][['product','category']])
cdf.loc[wrong_mask, 'category'] = 'Accessories'
print('After fix:Unique categories', cdf['category'].unique())

Before standardization: ['Ramesh Kumar' 'Priya Nair' 'AMIT VERMA' 'Sunita Patel' 'kiran mehta'
 'Deepak Singh']

 After Standardization: ['Ramesh Kumar' 'Priya Nair' 'Amit Verma' 'Sunita Patel' 'Kiran Mehta'
 'Deepak Singh']

 Before Keyboard rows with electronics category:
Empty DataFrame
Columns: [product, category]
Index: []
After fix:Unique categories ['Electronics' 'Accessories' 'Uncategorized']


In [ ]:
df['quantity']=pd.to_numeric(cdf['quantity'],errors='coerce')
df['unit_price']=pd.to_numeric(cdf['unit_price'],errors='coerce')
df['revenue']=df['quantity']*df['unit_price']
print('Revenue column created:')
print(df[['customer_name','product','quantity','unit_price','revenue']].head())
print(f"\n Total Revenue across all orders: rs{df['revenue'].sum():,.0f}")

Revenue column created:
  customer_name   product  quantity  unit_price  revenue
0  Ramesh Kumar    Laptop       2.0       45000  90000.0
1    Priya Nair       NaN       1.0       15000  15000.0
2    AMIT VERMA  Keyboard       3.0        1200   3600.0
3  Sunita Patel   Monitor       2.0       22000  44000.0
4  Ramesh Kumar    Laptop       2.0       45000  90000.0

 Total Revenue across all orders: rs818,000


In [ ]:
print('=' * 55)
print('   POST-CLEANING VALIDATION REPORT')
print('=' * 55)
print(f"Orginaal Rows : {len(df)}(duplicates)")
print(f"Cleaned Rows : {len(cdf)}")
print(f"Rows Removed : {len(df) - len(cdf)}(duplicates)")
print(f"Missing Values : {cdf.isnull().sum().sum()}")
print(f"Duplicates : {cdf.duplicated().sum()}")
print(f"Date Nulls : {cdf['order_date'].isnull().sum()}")
print(f"Revenue Nan : {df['revenue'].isnull().sum()}")
print(f"Categories : {sorted(cdf["category"].unique())}")
print('=' *55)

all_clean = (
    df.isnull().sum().sum()==0 and
    df.duplicated().sum()==0
)
print(f'Data is clean :{all_clean}')

   POST-CLEANING VALIDATION REPORT
Orginaal Rows : 30(duplicates)
Cleaned Rows : 30
Rows Removed : 0(duplicates)
Missing Values : 9
Duplicates : 0
Date Nulls : 2
Revenue Nan : 0
Categories : ['Accessories', 'Electronics', 'Uncategorized']
Data is clean :False


In [ ]:
API_KEY='f939f252c8d8f91a9b79a47ac57584a4'
BAASE_URL='https://home.openweathermap.org/api_keys'
CITIES=['Mumbai','Delhi','Bangalore','Chennai','Hyderabad','Kolkata','Pune','Jaipur']
print(f'API configured for {len(CITIES)} cities')
print(f'Cities:{CITIES}')
print('\nIMPORTANT:Replace YOUR_API_KEY with your own API key')

API configured for 8 cities
Cities:['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Jaipur']

IMPORTANT:Replace YOUR_API_KEY with your own API key


1. What are the three stages of ETL? Describe each stage using an example from today's sales dataset.
2. A DataFrame has 500 rows. After calling df.dropna() , it has 412 rows. What does this tell you?
3. Write code to remove duplicates from df where 'same row' means same customer_name AND same product.
4. What is the difference between fillna(p) and fillna(df['col'].median()) ? When would you prefer each?
5. Write Python code to call the weather API for 'Delhi' and print the temperature in Celsius.
6. What does response.status_code == 200 mean? What should you do when the code is 401?